<a href="https://colab.research.google.com/github/samueluribe27/Calidad_y_mineria_de_datos/blob/main/02_mineria_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
df = pd.read_csv('datos_limpios.csv')
df.head()

Saving datos_limpios.csv to datos_limpios.csv


,vigencia,ID_acuerdo,fec_sanc,titulo,articulado,tema,alcalde,pre_concejo,estado_acuerdo,obs_vigencia
0,2008,AD-001-2008,2008-03-13,Por medio se adiciona el acuerdo 024 de diciem...,2,Comportamiento ciudadano,Judith Del Carmen Pinedo Florez,Alberto Bernal Jimenez,Derogado,Derogado por el acuerdo 195 de 2025
1,2008,AD-002-2008,2008-06-05,Por medio del cual se adopta el plan de desarr...,74,Plan de Desarrollo,Judith Del Carmen Pinedo Florez,Alberto Bernal Jimenez,Vigente,No aplica
2,2008,AD-003-2008,2008-06-11,Por el cual se modifica el acuerdo 048 de 03 d...,4,Gestion administrativa,Judith Del Carmen Pinedo Florez,Alberto Bernal Jimenez,Vigente,No aplica
3,2008,AD-004-2008,2008-07-30,Por el cual se autoriza a la alcaldesa mayor d...,5,Cesion de predios o bienes,Judith Del Carmen Pinedo Florez,Alberto Bernal Jimenez,Derogado,Derogado por el acuerdo 195 de 2025
4,2008,AD-005-2008,2008-08-12,Por medio de la cual se crea el sistema integr...,5,Educacion,Judith Del Carmen Pinedo Florez,Alberto Bernal Jimenez,Vigente,No aplica


In [3]:
from sklearn.preprocessing import LabelEncoder

df_model = df.copy()

le = LabelEncoder()
df_model['tema_enc'] = le.fit_transform(df_model['tema'])
df_model['alcalde_enc'] = le.fit_transform(df_model['alcalde'])
df_model['vigencia_enc'] = df_model['vigencia']

df_model['objetivo'] = (df_model['estado_acuerdo'] == 'Vigente').astype(int)

X = df_model[['vigencia_enc', 'articulado', 'tema_enc', 'alcalde_enc']]
y = df_model['objetivo']

print("Variables de entrada:", X.columns.tolist())
print("Distribución objetivo:", y.value_counts())

Variables de entrada: ['vigencia_enc', 'articulado', 'tema_enc', 'alcalde_enc']
Distribución objetivo: objetivo
1    229
0    195
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelos = {
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Red Neuronal": MLPClassifier(max_iter=500, random_state=42),
    "SVM": SVC(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, pred)
    resultados[nombre] = acc
    print(f"\n{'='*40}")
    print(f"{nombre} — Accuracy: {acc:.4f}")
    print(classification_report(y_test, pred, target_names=['Derogado', 'Vigente']))

mejor = max(resultados, key=resultados.get)
print(f"\nMejor modelo: {mejor} con accuracy {resultados[mejor]:.4f}")


Árbol de Decisión — Accuracy: 0.8706
              precision    recall  f1-score   support

    Derogado       0.87      0.85      0.86        40
     Vigente       0.87      0.89      0.88        45

    accuracy                           0.87        85
   macro avg       0.87      0.87      0.87        85
weighted avg       0.87      0.87      0.87        85


KNN — Accuracy: 0.7529
              precision    recall  f1-score   support

    Derogado       0.71      0.80      0.75        40
     Vigente       0.80      0.71      0.75        45

    accuracy                           0.75        85
   macro avg       0.76      0.76      0.75        85
weighted avg       0.76      0.75      0.75        85


Red Neuronal — Accuracy: 0.5647
              precision    recall  f1-score   support

    Derogado       0.52      0.88      0.65        40
     Vigente       0.72      0.29      0.41        45

    accuracy                           0.56        85
   macro avg       0.62      0.58

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Random Forest — Accuracy: 0.8706
              precision    recall  f1-score   support

    Derogado       0.85      0.88      0.86        40
     Vigente       0.89      0.87      0.88        45

    accuracy                           0.87        85
   macro avg       0.87      0.87      0.87        85
weighted avg       0.87      0.87      0.87        85


Mejor modelo: Árbol de Decisión con accuracy 0.8706


In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print("Mejores hiperparámetros:", grid.best_params_)
print(f"Mejor accuracy con GridSearch: {grid.best_score_:.4f}")

mejor_modelo = grid.best_estimator_

Mejores hiperparámetros: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
Mejor accuracy con GridSearch: 0.8144


In [6]:
import pickle

with open('mejor_modelo.pkl', 'wb') as f:
    pickle.dump(mejor_modelo, f)

with open('label_encoders.pkl', 'wb') as f:
    pickle.dump({'tema': df_model['tema_enc'].values, 'temas_originales': df['tema'].unique()}, f)

files.download('mejor_modelo.pkl')
print("Modelo guardado")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Modelo guardado
